## Gold — `dim_situacao` (situação da obra)

**Origem:** `workspace.silver.cno` → **Destino:** `workspace.gold.dim_situacao`

- **Modelo:** Star Schema. A coluna `situacao` da silver vira dimensão própria; a fato referencia via `sk_situacao`.
- **Grão:** 1 linha por situação distinta presente na silver (código oficial RFB).
- **Transformações:**
  - Extração dos códigos distintos de `situacao`.
  - `descricao` decodificada via domínio oficial (01-NULA, 02-ATIVA, 03-SUSPENSA, 14-PARALISADA, 15-ENCERRADA).
  - `sk_situacao` = surrogate key sequencial (`row_number` ordenado por `codigo_situacao`, determinística).
- **Linhagem:** CSV dados.gov.br → `bronze.cno` → `silver.cno` → `gold.dim_situacao`.

In [0]:
%run ./_setup_cno

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from data_pipeline import save_table, add_column_comments
from metadata.metadata import DIM_SITUACAO_COMMENTS, DOMINIO_SITUACAO

In [0]:
SOURCE_TABLE = "workspace.silver.cno"
TARGET_TABLE = "workspace.gold.dim_situacao"

# Contrato de saída gold.dim_situacao
COLUNAS_ORDENADAS = [
    "sk_situacao",
    "codigo_situacao",
    "descricao",
]

In [0]:
df = spark.table(SOURCE_TABLE).select("situacao").distinct()
print(f"Situações distintas na silver: {df.count()}")

# Decodificação do código via domínio oficial RFB
df = df.select(
    F.col("situacao").alias("codigo_situacao"),
)
descricao_expr = None
for codigo, descricao in DOMINIO_SITUACAO.items():
    cond = F.col("codigo_situacao") == codigo
    descricao_expr = F.when(cond, F.lit(descricao)) if descricao_expr is None else descricao_expr.when(cond, F.lit(descricao))
df = df.withColumn("descricao", descricao_expr)

# Surrogate key determinística: ordenação pelo código oficial
w = Window.orderBy("codigo_situacao")
df = df.withColumn("sk_situacao", F.row_number().over(w).cast("int"))

df = df.select(*COLUNAS_ORDENADAS)
display(df)

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    DIM_SITUACAO_COMMENTS
)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
distintos_sk = spark.table(TARGET_TABLE).select("sk_situacao").distinct().count()
distintos_nk = spark.table(TARGET_TABLE).select("codigo_situacao").distinct().count()
sem_descricao = spark.table(TARGET_TABLE).filter(F.col("descricao").isNull()).count()
print(f"Total: {total:,} | SK distintos: {distintos_sk:,} | NK distintos: {distintos_nk:,} | Sem descrição: {sem_descricao}")
assert total == distintos_sk == distintos_nk and sem_descricao == 0, "Quebra de unicidade/descrição em dim_situacao"
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))